In [10]:
#Import the required packages
#Import package pandas for data analysis
import pandas as pd


In [11]:
def build_dqr_table(df, columns):
    """
    Build and save a Data Quality Report table.
    
    Parameters:
    -----------
    df : pandas.DataFrame
    columns : Index
        Features to include
    csv_filename : str
        Output filename
    
    Returns:
    --------
    pandas.DataFrame
        Complete DQR table
    """
    stats = df[columns].describe().T
    perc_missing = 100 * (df[columns].isnull().sum() / len(df))
    cardinality = df[columns].nunique()
    
    dqr_table = pd.concat([
        stats,
        pd.DataFrame(perc_missing, columns=['%missing']),
        pd.DataFrame(cardinality, columns=['cardinality'])
    ], axis=1)
    
    return dqr_table


print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [12]:
def load_data(filename):
    # Reading from a csv file, into a data frame with error handling
    try:
        # If the csv file has any special symbols, it helps to specify the file encoding.
        df = pd.read_csv(filename, 
                        keep_default_na=True, 
                        delimiter=',', 
                        skipinitialspace=True)
        
        print(f"✓ Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
        
        # Show the first few rows in the data frame
        display(df.head(5))
        
    except FileNotFoundError:
        print("ERROR: The file 'ppr-group-25204989-train.csv' was not found.")
        print("Please ensure the file is in the current working directory.")
    except Exception as e:
        print(f"ERROR: An unexpected error occurred: {e}")

    return df

In [13]:
# Validate that the dataframe has data and expected columns
def validate_data(df):
    print("Dataset Validation Checks:")
    print("="*60)

    # Check 1: Dataset is not empty
    if df.empty:
        print("⚠ WARNING: Dataset is empty!")
    else:
        print(f"✓ Dataset contains {df.shape[0]:,} rows")

    # Check 2: Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"⚠ WARNING: Found {duplicate_count} duplicate rows")
    else:
        print("✓ No duplicate rows found")

    # Check 3: Check if all columns are unnamed
    unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
    if unnamed_cols:
        print(f"⚠ WARNING: Found {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
    else:
        print("✓ All columns have names")

    # Check 4: Basic column overview
    print(f"\n✓ Dataset has {df.shape[1]} columns:")
    print(f"  - Column names: {list(df.columns)}")

## 1. ALF01.csv

In [14]:
df=load_data("MUM01.csv")

✓ Successfully loaded dataset with 6192 rows and 10 columns


,STATISTIC,Statistic Label,TLIST(M1),Month,C02076V02508,Age Group,C02199V02655,Sex,UNIT,VALUE
0,MUM01C01,Seasonally Adjusted Monthly Unemployment,199801,1998 January,310,15 - 24 years,-,Both sexes,Thousand,55.0
1,MUM01C01,Seasonally Adjusted Monthly Unemployment,199801,1998 January,310,15 - 24 years,1,Male,Thousand,31.3
2,MUM01C01,Seasonally Adjusted Monthly Unemployment,199801,1998 January,310,15 - 24 years,2,Female,Thousand,23.7
3,MUM01C01,Seasonally Adjusted Monthly Unemployment,199801,1998 January,316,15 - 74 years,-,Both sexes,Thousand,151.0
4,MUM01C01,Seasonally Adjusted Monthly Unemployment,199801,1998 January,316,15 - 74 years,1,Male,Thousand,86.7


In [15]:
validate_data(df)

Dataset Validation Checks:
✓ Dataset contains 6,192 rows
✓ No duplicate rows found
✓ All columns have names

✓ Dataset has 10 columns:
  - Column names: ['STATISTIC', 'Statistic Label', 'TLIST(M1)', 'Month', 'C02076V02508', 'Age Group', 'C02199V02655', 'Sex', 'UNIT', 'VALUE']


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6192 entries, 0 to 6191
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   STATISTIC        6192 non-null   object 
 1   Statistic Label  6192 non-null   object 
 2   TLIST(M1)        6192 non-null   int64  
 3   Month            6192 non-null   object 
 4   C02076V02508     6192 non-null   int64  
 5   Age Group        6192 non-null   object 
 6   C02199V02655     6192 non-null   object 
 7   Sex              6192 non-null   object 
 8   UNIT             6192 non-null   object 
 9   VALUE            6192 non-null   float64
dtypes: float64(1), int64(2), object(7)
memory usage: 483.9+ KB


In [22]:


# Convert all object columns to categorical
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype('category')

df['Month']=df['Month'].astype('category')
df['TLIST(M1)']=df['TLIST(M1)'].astype('category')
df['C02076V02508']=df['C02076V02508'].astype('category')

# Update column groups after type corrections
numeric_columns = df.select_dtypes(['int64', 'float64']).columns
category_columns = df.select_dtypes('category').columns

print("✓ Data types corrected:")
print(f"  - Numeric features: {len(numeric_columns)}")
print(f"  - Categorical features: {len(category_columns)}")

✓ Data types corrected:
  - Numeric features: 1
  - Categorical features: 9


In [23]:
df_numeric_dqr = build_dqr_table(df, numeric_columns)
display(df_numeric_dqr)

,count,mean,std,min,25%,50%,75%,max,%missing,cardinality
VALUE,6192.0,41.718621,55.025435,2.9,7.475,19.0,54.3,361.0,0.0,1414


In [24]:
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
STATISTIC,6192,2,MUM01C01,3096,0.0,2
Statistic Label,6192,2,Seasonally Adjusted Monthly Unemployment,3096,0.0,2
TLIST(M1),6192,344,199801,18,0.0,344
Month,6192,344,1998 April,18,0.0,344
C02076V02508,6192,3,310,2064,0.0,3
Age Group,6192,3,15 - 24 years,2064,0.0,3
C02199V02655,6192,3,-,2064,0.0,3
Sex,6192,3,Both sexes,2064,0.0,3
UNIT,6192,2,%,3096,0.0,2


In [27]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


STATISTIC


,value,count
0,MUM01C01,3096
1,MUM01C02,3096



Statistic Label


,value,count
0,Seasonally Adjusted Monthly Unemployment,3096
1,Seasonally Adjusted Monthly Unemployment Rate,3096



TLIST(M1)


...

,value,count
0,-,2064
1,1,2064
2,2,2064



Sex


,value,count
0,Both sexes,2064
1,Female,2064
2,Male,2064



UNIT


,value,count
0,%,3096
1,Thousand,3096
